# 📚 Column Creation & Value Transformation
### 열 생성 & 값 변환

> **Section 6 of 11** · Pandas Complete Reference Guide for a JS/TS developer transitioning into BA   
> 전체 11개 섹션 중 **6번째** · JS/TS 개발자 출신 BA를 위한 Pandas 완전 참조 가이드

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배울 내용:
- [x] How to add new columns by direct assignment or with `assign()`, and branch values with `np.where` / `np.select`  
직접 대입 또는 `assign()`으로 새 열을 추가하고, `np.where` / `np.select`로 값을 분기하는 방법
- [x] How to bin continuous values with `pd.cut` / `pd.qcut`, and apply custom logic with `apply()`  
`pd.cut` / `pd.qcut`으로 연속값을 구간화하고, `apply()`로 커스텀 로직을 적용하는 방법
- [x] How to map/replace values, one-hot encode with `get_dummies`, and chain transformations with `pipe()` / `eval()`  
값을 map/replace하고, `get_dummies`로 원-핫 인코딩하고, `pipe()` / `eval()`로 변환을 체이닝하는 방법

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**English**
This is where you go from "the columns I was given" to "the columns the analysis actually needs" — computing a new metric (`revenue = price × quantity`), branching a value into a category (pass/fail, A/B/C grade), bucketing a continuous number into a range (age group, spending tier), or converting a code into a readable label. Every technique here builds *on top of* clean data from Section 5 — it doesn't fix data, it derives new meaning from it.

**한글**
이는 "받은 열"에서 "분석에 실제로 필요한 열"로 넘어가는 단계입니다 — 새 지표를 계산하거나(`revenue = price × quantity`), 값을 카테고리로 분기하거나(합격/불합격, A/B/C 등급), 연속된 숫자를 구간으로 나누거나(연령대, 소비 등급), 코드를 읽기 쉬운 라벨로 변환합니다. 여기 나오는 모든 기법은 5번 섹션에서 정제된 데이터 *위에* 세워집니다 — 데이터를 고치는 게 아니라, 거기서 새로운 의미를 끌어냅니다.

## Why do we use it?
*(When is it useful?)*

**English**
A raw table rarely already has the exact column a chart or report needs — "profit margin," "customer segment," "is this a repeat order" are almost always *derived*, not given. Doing this derivation once, correctly, and by whole column (vectorized) instead of row-by-row is what keeps a growing dataset fast.

**한글**
원본 테이블에는 차트나 보고서가 필요로 하는 바로 그 열이 이미 들어있는 경우가 드뭅니다 — "이익률", "고객 세그먼트", "재구매 여부"는 거의 항상 *파생*된 것이지, 원래 주어진 것이 아닙니다. 이런 파생 작업을 한 번에, 정확하게, 행 단위가 아니라 열 전체 단위(벡터 연산)로 하는 것이 데이터셋이 커져도 빠르게 유지되는 비결입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**English**
This is the bridge between "raw numbers" and "an insight a stakeholder can act on" — turning a `spend` column into a `spend_tier` is the difference between a scatterplot nobody reads and a segment a marketing campaign can actually target. It's also usually the last step before GroupBy and visualization.

**한글**
이는 "원시 숫자"와 "이해관계자가 실행에 옮길 수 있는 인사이트" 사이의 다리입니다 — `spend` 열을 `spend_tier`로 바꾸는 것은, 아무도 안 읽는 산점도와 마케팅 캠페인이 실제로 타겟팅할 수 있는 세그먼트의 차이를 만듭니다. 이는 또한 보통 GroupBy와 시각화 직전의 마지막 단계입니다.

### Quick Comparison: JS/TS vs pandas / 빠른 비교

| Concept / 개념 | JavaScript / TypeScript | pandas |
|---|---|---|
| Add a computed field / 계산된 필드 추가 | `arr.map(o => ({...o, revenue: o.price*o.qty}))` | `df["revenue"] = df["price"] * df["qty"]` |
| Ternary branch / 삼항 분기 | `x >= 60 ? "Pass" : "Fail"` | `np.where(x >= 60, "Pass", "Fail")` |
| Multi-branch (switch-like) / 다중 분기 (switch와 유사) | `if / else if` chain / `if / else if` 체인 | `np.select([...], [...], default=...)` |
| Bucket a number into a range / 숫자를 구간으로 분류 | manual `if` chain / 직접 `if` 체인 | `pd.cut()` / `pd.qcut()` |
| Look up a value in a dict / dict에서 값 조회 | `map[key] ?? fallback` | `series.map(dict)` / `.replace(dict)` |
| One-hot encode a category / 범주 원-핫 인코딩 | manual loop building flag fields / 직접 반복문으로 플래그 필드 생성 | `pd.get_dummies()` |
| Chain custom transform functions / 커스텀 변환 함수 체이닝 | `.then()` chain / functional pipe | `df.pipe(fn1).pipe(fn2)` |

---
# 📝 Syntax

## Basic Syntax

In [1]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard"],
    "price": [1200000, 25000, 45000],
    "quantity": [3, 20, 12],
})

# Direct assignment -- the simplest way to add a computed column
# 직접 대입 -- 계산된 열을 추가하는 가장 단순한 방법
sales["revenue"] = sales["price"] * sales["quantity"]
print(sales)
print()

# np.where -- a vectorized ternary / np.where -- 벡터화된 삼항 연산
sales["order_size"] = np.where(sales["quantity"] >= 15, "Bulk", "Regular")
print(sales)

    product    price  quantity  revenue
0    Laptop  1200000         3  3600000
1     Mouse    25000        20   500000
2  Keyboard    45000        12   540000

    product    price  quantity  revenue order_size
0    Laptop  1200000         3  3600000    Regular
1     Mouse    25000        20   500000       Bulk
2  Keyboard    45000        12   540000    Regular


## Common Variations

In [2]:
import pandas as pd

sales = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard"],
    "price": [1200000, 25000, 45000],
    "quantity": [3, 20, 12],
})

# assign() -- returns a NEW DataFrame, so it chains cleanly / assign() -- 새 DataFrame을 반환, 체이닝에 깔끔
result = sales.assign(revenue=lambda x: x["price"] * x["quantity"])
print(result)
print()

# map() -- look up each value in a dict / map() -- 각 값을 dict에서 조회
size_map = {"Laptop": "Large", "Mouse": "Small", "Keyboard": "Medium"}
sales["size"] = sales["product"].map(size_map)
print(sales)

    product    price  quantity  revenue
0    Laptop  1200000         3  3600000
1     Mouse    25000        20   500000
2  Keyboard    45000        12   540000

    product    price  quantity    size
0    Laptop  1200000         3   Large
1     Mouse    25000        20   Small
2  Keyboard    45000        12  Medium


---
# 🧪 Small Examples

## Example 1 — Direct Assignment vs assign()
*(Covers source sections 6-1 and 6-2)*

**English:** `df["col"] = ...` is the simplest way to add a column, and later lines can freely reference the column it just created. `df.assign(col=lambda x: ...)` does the same thing but *returns a new DataFrame* instead of mutating the original — which is exactly what makes it chain cleanly into a `filter → add columns → sort` pipeline.  
**한글:** `df["col"] = ...`는 열을 추가하는 가장 단순한 방법이며, 이후 줄에서 방금 만든 열을 자유롭게 참조할 수 있습니다. `df.assign(col=lambda x: ...)`도 같은 일을 하지만 원본을 수정하는 대신 *새 DataFrame을 반환*합니다 — 바로 이 점 때문에 `필터 → 열 추가 → 정렬` 파이프라인에 깔끔하게 체이닝됩니다.

| | Direct assignment / 직접 대입 | `.assign()` |
|---|---|---|
| Chainable / 체이닝 가능 | No / 불가 | Yes / 가능 |
| Modifies the original / 원본 수정 | Yes, in place / 예, 즉시 수정 | No, returns a new DataFrame / 아니오, 새 DataFrame 반환 |
| Referencing a column just added / 방금 추가한 열 참조 | a separate line / 별도의 줄 | `lambda x: x["col"]`, same call / 같은 호출 안에서 |
| Best for / 적합한 상황 | quick, standalone scripts / 빠르고 단독적인 스크립트 | pipelines: filter → add columns → sort / 필터 → 열 추가 → 정렬 파이프라인 |

In [3]:
import pandas as pd

# Direct assignment -- straightforward, mutates sales in place
# 직접 대입 -- 단순함, sales를 즉시 수정함
sales = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "price": [1200000, 25000, 45000, 350000],
    "cost": [900000, 15000, 30000, 250000],
    "quantity": [3, 20, 12, 5],
})
sales["revenue"] = sales["price"] * sales["quantity"]
sales["profit"] = (sales["price"] - sales["cost"]) * sales["quantity"]
sales["margin"] = (sales["profit"] / sales["revenue"] * 100).round(1)
print("direct assignment:")
print(sales)
print()

# assign() -- same result, but chainable and referencing earlier lambdas inline
# assign() -- 결과는 같지만 체이닝 가능하고 이전 lambda를 같은 호출 안에서 참조
sales2 = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "price": [1200000, 25000, 45000, 350000],
    "cost": [900000, 15000, 30000, 250000],
    "quantity": [3, 20, 12, 5],
})
result = sales2.assign(
    revenue = lambda x: x["price"] * x["quantity"],
    profit = lambda x: (x["price"] - x["cost"]) * x["quantity"],
    margin_pct = lambda x: (x["profit"] / x["revenue"] * 100).round(1),
)
print("assign():")
print(result)

direct assignment:
    product    price    cost  quantity  revenue  profit  margin
0    Laptop  1200000  900000         3  3600000  900000    25.0
1     Mouse    25000   15000        20   500000  200000    40.0
2  Keyboard    45000   30000        12   540000  180000    33.3
3   Monitor   350000  250000         5  1750000  500000    28.6

assign():
    product    price    cost  quantity  revenue  profit  margin_pct
0    Laptop  1200000  900000         3  3600000  900000        25.0
1     Mouse    25000   15000        20   500000  200000        40.0
2  Keyboard    45000   30000        12   540000  180000        33.3
3   Monitor   350000  250000         5  1750000  500000        28.6


## Example 2 — np.where / np.select: Branching Values
*(Covers source section 6-3)*

**English:** `np.where(condition, if_true, if_false)` is a vectorized ternary for exactly two outcomes. `np.select([cond1, cond2, ...], [choice1, choice2, ...], default=...)` handles three or more outcomes — conditions are checked **top to bottom, and the first match wins**, with `default` covering anything that matches none of them.  
**한글:** `np.where(condition, if_true, if_false)`는 정확히 두 가지 결과를 위한 벡터화된 삼항 연산입니다. `np.select([cond1, cond2, ...], [choice1, choice2, ...], default=...)`는 세 가지 이상의 결과를 처리합니다 — 조건은 **위에서 아래로 순서대로 검사되고, 첫 번째로 일치하는 것이 이깁니다**. `default`는 어느 조건에도 해당하지 않는 경우를 처리합니다.

In [4]:
import pandas as pd
import numpy as np

scores = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "score": [92, 45, 73, 58, 88],
})

# np.where -- exactly two outcomes / np.where -- 정확히 두 가지 결과
scores["pass_fail"] = np.where(scores["score"] >= 60, "Pass", "Fail")
print(scores)
print()

# np.select -- three or more outcomes, checked top to bottom / np.select -- 세 가지 이상, 위에서 아래로 검사
conditions = [
    scores["score"] >= 90,
    scores["score"] >= 80,
    scores["score"] >= 70,
    scores["score"] >= 60,
]
choices = ["A", "B", "C", "D"]
scores["grade"] = np.select(conditions, choices, default="F")
print(scores)

      name  score pass_fail
0    Alice     92      Pass
1      Bob     45      Fail
2  Charlie     73      Pass
3    Diana     58      Fail
4      Eve     88      Pass

      name  score pass_fail grade
0    Alice     92      Pass     A
1      Bob     45      Fail     F
2  Charlie     73      Pass     C
3    Diana     58      Fail     F
4      Eve     88      Pass     B


## Example 3 — pd.cut / pd.qcut: Binning Continuous Values
*(Covers source section 6-4)*

**English:** `pd.cut()` splits a numeric column using **fixed boundaries you choose** (`bins=[20,30,40,50,70]`) — every bin can hold a different number of rows. `pd.qcut()` splits by **quantile** (`q=4` for quartiles) so every bin holds roughly the **same number** of rows instead — the boundaries are calculated for you.  
**한글:** `pd.cut()`은 **직접 정한 고정 경계값**(`bins=[20,30,40,50,70]`)으로 숫자 열을 나눕니다 — 각 구간의 인원수는 서로 다를 수 있습니다. `pd.qcut()`은 **분위수**(`q=4`는 4분위)로 나눠서 각 구간이 대략 **같은 개수**의 행을 갖게 합니다 — 경계값은 자동으로 계산됩니다.

### `cut` vs `qcut` / 언제 무엇을 쓸까

| | `pd.cut` | `pd.qcut` |
|---|---|---|
| Boundary basis / 경계 기준 | value range you specify / 직접 지정한 값의 범위 | frequency — equal-sized groups / 빈도 — 그룹 크기가 동일 |
| Bin width / 구간 폭 | can be equal or not, your choice / 동일하거나 다를 수 있음, 직접 선택 | usually unequal / 보통 다름 |
| Rows per bin / 구간별 인원 | can vary a lot / 크게 다를 수 있음 | roughly equal / 거의 동일 |
| Typical use / 주 용도 | "20s / 30s / 40s" age bands / "20대/30대/40대" 같은 연령대 | "top 25% / bottom 25%" percentile scoring / "상위 25%/하위 25%" 같은 백분위 스코어링 |

In [5]:
import pandas as pd

customers = pd.DataFrame({
    "name": list("ABCDEFGH"),
    "age": [22, 35, 28, 45, 52, 31, 67, 41],
})

# pd.cut -- fixed boundaries you choose / pd.cut -- 직접 정한 고정 경계값
customers["age_group"] = pd.cut(
    customers["age"],
    bins=[20, 30, 40, 50, 70],
    labels=["20s", "30s", "40s", "50s+"],
    right=False   # [20,30) includes the left edge, excludes the right / 왼쪽 포함, 오른쪽 미포함
)
print("pd.cut -- fixed age bands:")
print(customers)
print()

spend_data = pd.DataFrame({
    "name": list("ABCDEFGH"),
    "spend": [120000, 450000, 85000, 320000, 680000, 210000, 95000, 510000],
})

# pd.qcut -- quantile-based, roughly equal group sizes / pd.qcut -- 분위수 기반, 그룹 크기가 거의 동일
spend_data["spend_tier"] = pd.qcut(
    spend_data["spend"],
    q=4,
    labels=["Bottom 25%", "Mid-low", "Mid-high", "Top 25%"]
)
print("pd.qcut -- spend quartiles:")
print(spend_data)

pd.cut -- fixed age bands:
  name  age age_group
0    A   22       20s
1    B   35       30s
2    C   28       20s
3    D   45       40s
4    E   52      50s+
5    F   31       30s
6    G   67      50s+
7    H   41       40s

pd.qcut -- spend quartiles:
  name   spend  spend_tier
0    A  120000     Mid-low
1    B  450000    Mid-high
2    C   85000  Bottom 25%
3    D  320000    Mid-high
4    E  680000     Top 25%
5    F  210000     Mid-low
6    G   95000  Bottom 25%
7    H  510000     Top 25%


## Example 4 — apply(): Custom Row/Column Logic
*(Covers source section 6-5)*

**English:** `Series.apply(fn)` runs `fn` on every value in a column, one at a time — useful for string cleanup a vectorized method can't express directly. `DataFrame.apply(fn, axis=1)` runs `fn` once per **row**, giving the function access to *every column of that row at once* — but it's dramatically slower than a vectorized operation, so it's a last resort, not a first instinct.  
**한글:** `Series.apply(fn)`은 열의 모든 값에 대해 `fn`을 하나씩 실행합니다 — 벡터화된 메서드로 직접 표현할 수 없는 문자열 정리 등에 유용합니다. `DataFrame.apply(fn, axis=1)`은 **행**마다 `fn`을 한 번씩 실행해서, 함수가 *그 행의 모든 열에 한 번에* 접근할 수 있게 합니다 — 하지만 벡터 연산보다 훨씬 느리므로, 첫 번째 선택지가 아니라 최후의 수단입니다.

In [6]:
import pandas as pd
import numpy as np
import time

# Series.apply -- element-wise text cleanup / Series.apply -- 요소별 텍스트 정리
orders = pd.DataFrame({
    "product": ["laptop_pro", "mouse_basic", "keyboard_mech", "monitor_4k"],
    "price": [1200000, 25000, 45000, 350000],
})
orders["product_clean"] = orders["product"].apply(lambda x: x.replace("_", " ").title())
print("Series.apply:")
print(orders)
print()

# DataFrame.apply(axis=1) -- needs several columns of the SAME row at once
# DataFrame.apply(axis=1) -- 같은 행의 여러 열이 동시에 필요할 때
orders["region"] = ["Seoul", "Busan", "Seoul", "Incheon"]

def tax_rate(row):
    if row["region"] == "Seoul":
        return row["price"] * 0.10
    return row["price"] * 0.08

orders["tax"] = orders.apply(tax_rate, axis=1)
print("DataFrame.apply(axis=1):")
print(orders)
print()

# Why apply() is a last resort -- a real speed comparison on 100,000 rows
# apply()가 최후의 수단인 이유 -- 10만 행에 대한 실제 속도 비교
big = pd.DataFrame({"val": np.random.randint(0, 1000, 100_000)})
t0 = time.time(); big["v1"] = big["val"].apply(lambda x: x * 2); t1 = time.time()
big["v2"] = big["val"] * 2; t2 = time.time()
print(f"apply():     {(t1 - t0) * 1000:.0f} ms")
print(f"vectorized:  {(t2 - t1) * 1000:.0f} ms")
print("-> vectorized math is roughly 100x faster on this machine -- your exact numbers will vary")
print("-> 이 컴퓨터 기준 벡터 연산이 대략 100배 빠름 -- 정확한 수치는 환경마다 다를 수 있음")

Series.apply:
         product    price  product_clean
0     laptop_pro  1200000     Laptop Pro
1    mouse_basic    25000    Mouse Basic
2  keyboard_mech    45000  Keyboard Mech
3     monitor_4k   350000     Monitor 4K

DataFrame.apply(axis=1):
         product    price  product_clean   region       tax
0     laptop_pro  1200000     Laptop Pro    Seoul  120000.0
1    mouse_basic    25000    Mouse Basic    Busan    2000.0
2  keyboard_mech    45000  Keyboard Mech    Seoul    4500.0
3     monitor_4k   350000     Monitor 4K  Incheon   28000.0

apply():     32 ms
vectorized:  1 ms
-> vectorized math is roughly 100x faster on this machine -- your exact numbers will vary
-> 이 컴퓨터 기준 벡터 연산이 대략 100배 빠름 -- 정확한 수치는 환경마다 다를 수 있음


## Example 5 — map / replace: Value Lookups
*(Covers source section 6-6)*

**English:** `.map(dict)` looks up every value in a dict — anything **not found in the dict becomes `NaN`**, which makes it good for catching unexpected codes. `.replace(dict)` does the same lookup but **leaves unmapped values unchanged** instead of erasing them. Chaining `.map(dict).fillna("Other")` gives you an explicit fallback instead of a silent `NaN`.  
**한글:** `.map(dict)`는 모든 값을 dict에서 조회하는데 — **dict에 없는 값은 `NaN`이 됩니다** — 그래서 예상치 못한 코드를 잡아내는 데 유용합니다. `.replace(dict)`는 같은 조회를 하지만 **매핑되지 않은 값은 그대로 유지**합니다(지우지 않음). `.map(dict).fillna("Other")`를 체이닝하면 조용한 `NaN` 대신 명시적인 대체값을 얻을 수 있습니다.

In [7]:
import pandas as pd

orders = pd.DataFrame({
    "product_code": ["P01", "P02", "P03", "P04", "P05"],
    "status_code": ["A", "B", "A", "C", "X"],
})

code_map = {"P01": "Laptop", "P02": "Mouse", "P03": "Keyboard", "P04": "Monitor"}
status_map = {"A": "Active", "B": "Inactive", "C": "On Hold"}

# map -- unmapped values (P05 has no entry) become NaN / map -- 매핑 없는 값(P05)은 NaN이 됨
orders["product_name"] = orders["product_code"].map(code_map)

# replace -- unmapped values (status "X") are kept as-is / replace -- 매핑 없는 값(status "X")은 그대로 유지
orders["status"] = orders["status_code"].replace(status_map)
print(orders)
print()

# map + fillna -- an explicit fallback instead of a silent NaN
# map + fillna -- 조용한 NaN 대신 명시적인 대체값
orders["product_name"] = orders["product_code"].map(code_map).fillna("Other")
print(orders)

  product_code status_code product_name    status
0          P01           A       Laptop    Active
1          P02           B        Mouse  Inactive
2          P03           A     Keyboard    Active
3          P04           C      Monitor   On Hold
4          P05           X          NaN         X

  product_code status_code product_name    status
0          P01           A       Laptop    Active
1          P02           B        Mouse  Inactive
2          P03           A     Keyboard    Active
3          P04           C      Monitor   On Hold
4          P05           X        Other         X


## Example 6 — get_dummies: One-Hot Encoding
*(Covers source section 6-7)*

**English:** `pd.get_dummies()` turns a category column into several `True`/`False` columns, one per unique value — the standard way to represent text categories numerically for a model. `drop_first=True` drops the first category's column (it can always be inferred from the others being all `False`) to avoid redundant, perfectly-correlated columns.  
**한글:** `pd.get_dummies()`는 카테고리 열을 고유값마다 하나씩, 여러 개의 `True`/`False` 열로 바꿉니다 — 텍스트 카테고리를 모델을 위해 숫자로 표현하는 표준 방법입니다. `drop_first=True`는 첫 번째 카테고리의 열을 제거합니다(나머지가 모두 `False`이면 자동으로 유추 가능하므로) — 불필요하게 완벽히 상관된 열을 피하기 위함입니다.

In [8]:
import pandas as pd

customers = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "region": ["Seoul", "Busan", "Seoul", "Incheon"],
    "tier": ["Gold", "Silver", "Gold", "Bronze"],
})

print("get_dummies -- one column per unique value:")
print(pd.get_dummies(customers, columns=["region", "tier"]))
print()

# drop_first=True -- N categories only need N-1 columns / drop_first=True -- N개 카테고리는 N-1개 열이면 충분
print("get_dummies(drop_first=True) -- avoids redundant columns:")
print(pd.get_dummies(customers, columns=["region", "tier"], drop_first=True))

get_dummies -- one column per unique value:
      name  region_Busan  region_Incheon  region_Seoul  tier_Bronze  \
0    Alice         False           False          True        False   
1      Bob          True           False         False        False   
2  Charlie         False           False          True        False   
3    Diana         False            True         False         True   

   tier_Gold  tier_Silver  
0       True        False  
1      False         True  
2       True        False  
3      False        False  

get_dummies(drop_first=True) -- avoids redundant columns:
      name  region_Incheon  region_Seoul  tier_Gold  tier_Silver
0    Alice           False          True       True        False
1      Bob           False         False      False         True
2  Charlie           False          True       True        False
3    Diana            True         False      False        False


## Example 7 — pipe() & eval(): Chaining and Expression Syntax
*(Covers source sections 6-8 and 6-9)*

**English:** `.pipe(fn)` threads a DataFrame through a **custom, reusable function** as one step in a method chain — useful for cleaning steps you'll apply to many different datasets. `.eval("new_col = expr")` creates a column from a string expression, letting you skip repeating `df["..."]` for every column name — its main benefit is readability, not speed (the speed boost only applies with the optional `numexpr` library installed).  
**한글:** `.pipe(fn)`은 **커스텀 재사용 함수**를 메서드 체인의 한 단계로 통과시킵니다 — 여러 데이터셋에 반복 적용할 정제 단계에 유용합니다. `.eval("new_col = expr")`은 문자열 표현식으로 열을 만들어서, 모든 열 이름마다 `df["..."]`를 반복하지 않아도 됩니다 — 주된 장점은 속도가 아니라 가독성입니다(속도 이점은 선택적 라이브러리인 `numexpr`가 설치된 환경에서만 적용됩니다).

In [9]:
import pandas as pd
from io import StringIO

# pipe -- reusable cleaning functions, chained / pipe -- 재사용 가능한 정제 함수를 체이닝
raw_csv = '''name,dept,salary
 alice ,SalesTeam,48000000
BOB,MarketingTeam,35000000
charlie,EngineeringTeam,62000000'''

def clean_names(df):
    df["name"] = df["name"].str.strip().str.title()
    return df

def remove_suffix(df, col, suffix):
    df[col] = df[col].str.replace(suffix, "", regex=False)
    return df

df = (
    pd.read_csv(StringIO(raw_csv))
    .pipe(clean_names)
    .pipe(remove_suffix, col="dept", suffix="Team")
)
print("pipe():")
print(df)
print()

# eval -- add columns via a string expression, no repeated df["..."] / eval -- 문자열 표현식으로 열 추가
sales = pd.DataFrame({
    "price": [1200000, 25000, 45000, 350000],
    "cost": [900000, 15000, 30000, 250000],
    "quantity": [3, 20, 12, 5],
})
sales.eval("revenue = price * quantity", inplace=True)
sales.eval("profit = (price - cost) * quantity", inplace=True)
sales["margin"] = (sales["profit"] / sales["revenue"] * 100).round(1)
print("eval():")
print(sales)

pipe():
      name         dept    salary
0    Alice        Sales  48000000
1      Bob    Marketing  35000000
2  Charlie  Engineering  62000000

eval():
     price    cost  quantity  revenue  profit  margin
0  1200000  900000         3  3600000  900000    25.0
1    25000   15000        20   500000  200000    40.0
2    45000   30000        12   540000  180000    33.3
3   350000  250000         5  1750000  500000    28.6


## Example 8 — Common Combo Patterns
*(Covers source section 6-10)*

**English:** Pattern A chains `assign()` + `pd.cut()` + `query()` to bucket a value and filter in one flow. Pattern B chains `assign()` + `map()` + `pd.cut()` + `np.where()` to translate codes, bucket a value, and flag a condition — three new columns built from a single method call.   
**한글:** 패턴 A는 `assign()` + `pd.cut()` + `query()`를 체이닝해서 값을 구간화하고 한 흐름으로 필터링합니다. 패턴 B는 `assign()` + `map()` + `pd.cut()` + `np.where()`를 체이닝해서 코드를 변환하고, 값을 구간화하고, 조건을 표시합니다 — 단일 메서드 호출로 새 열 3개를 만듭니다.

In [10]:
import pandas as pd
import numpy as np

# Pattern A: assign + cut + query -- bucket, then filter, in one chain
# 패턴 A: assign + cut + query -- 구간화 후 한 번에 필터링
customers = pd.DataFrame({
    "name": list("ABCDEF"),
    "spend": [120000, 450000, 85000, 320000, 680000, 210000],
    "region": ["Seoul", "Busan", "Seoul", "Incheon", "Seoul", "Busan"],
})

result_a = (
    customers
    .assign(spend_tier=pd.cut(
        customers["spend"],
        bins=[0, 150000, 400000, float("inf")],
        labels=["Small", "Medium", "Large"]
    ))
    .query("region == 'Seoul'")
    .sort_values("spend", ascending=False)
)
print("Pattern A -- Seoul customers, bucketed by spend:")
print(result_a)
print()

# Pattern B: map + assign + np.where -- translate codes + bucket + flag, all at once
# 패턴 B: map + assign + np.where -- 코드 변환 + 구간화 + 플래그를 한 번에
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "region_code": ["SEO", "PUS", "ICN", "SEO"],
    "amount": [45000, 32000, 61000, 28000],
})
region_name = {"SEO": "Seoul", "PUS": "Busan", "ICN": "Incheon"}

result_b = orders.assign(
    region = lambda x: x["region_code"].map(region_name),
    tier = lambda x: pd.cut(x["amount"], bins=[0, 30000, 50000, float("inf")], labels=["Small", "Medium", "Large"]),
    is_seoul = lambda x: np.where(x["region"] == "Seoul", True, False),
)
print("Pattern B -- codes translated, bucketed, and flagged:")
print(result_b)

Pattern A -- Seoul customers, bucketed by spend:
  name   spend region spend_tier
4    E  680000  Seoul      Large
0    A  120000  Seoul      Small
2    C   85000  Seoul      Small

Pattern B -- codes translated, bucketed, and flagged:
   order_id region_code  amount   region    tier  is_seoul
0         1         SEO   45000    Seoul  Medium      True
1         2         PUS   32000    Busan  Medium     False
2         3         ICN   61000  Incheon   Large     False
3         4         SEO   28000    Seoul   Small      True


## Example 9 (Practice) — Fill in the Blanks
*(Based on the practice exercise in source section 6-11)*

**English:** Fill in each `________` blank below. The code is syntactically valid Python, so it won't raise a `SyntaxError` — but it also won't print any result until every blank is correct (it will raise a runtime error instead, which is expected).
**한글:** 아래 `________` 빈칸을 채워보세요. 코드는 문법적으로 올바른 파이썬이라 `SyntaxError`는 나지 않지만, 모든 빈칸이 정확해지기 전까지는 결과가 출력되지 않습니다(대신 런타임 오류가 나는데, 이는 의도된 동작입니다).

In [12]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Webcam"],
    "price": [1200000, 25000, 45000, 350000, 75000],
    "cost": [900000, 15000, 30000, 250000, 50000],
    "quantity": [3, 20, 12, 5, 8],
})

# 1. Add revenue (price x quantity) / revenue 추가 (price x quantity)
sales["revenue"] = sales["price"] * sales["quantity"]

# 2. Add profit ((price - cost) x quantity) / profit 추가 ((price - cost) x quantity)
sales["profit"] = (sales["price"] - sales["cost"]) * sales["quantity"]

# 3. Add margin (profit / revenue x 100, rounded to 1 decimal) / margin 추가 (profit / revenue x 100, 소수점 1자리)
sales["margin"] = (sales["profit"] / sales["revenue"] * 100).round(1)

# 4. Add profit_grade -- "High Margin" if margin >= 35, else "Standard"
#    profit_grade 추가 -- margin이 35 이상이면 "High Margin", 아니면 "Standard"
sales["profit_grade"] = np.where(sales["margin"] >= 35, "High Margin", "Standard")

# 5. Add revenue_tier via cut into 3 bands: 0-500,000 "Small", 500,001-2,000,000 "Medium", 2,000,001+ "Large"
#    revenue_tier 추가 -- cut으로 3구간: 0~500,000 "Small", 500,001~2,000,000 "Medium", 2,000,001+ "Large"
sales["revenue_tier"] = pd.cut(
    sales["revenue"],
    bins=[0, 500000, 2000000, float("inf")],
    labels=["Small", "Medium", "Large"]
)

print(sales[["product", "revenue", "profit", "margin", "profit_grade", "revenue_tier"]])

    product  revenue  profit  margin profit_grade revenue_tier
0    Laptop  3600000  900000    25.0     Standard        Large
1     Mouse   500000  200000    40.0  High Margin        Small
2  Keyboard   540000  180000    33.3     Standard       Medium
3   Monitor  1750000  500000    28.6     Standard       Medium
4    Webcam   600000  200000    33.3     Standard       Medium


### 💡 Hint / 힌트
`price` · `quantity` · `cost` · `profit` · `revenue` · `1` · `35`

### ✅ Solution / 정답
*(Try solving it yourself first! / 먼저 스스로 풀어본 뒤에 확인하세요!)*

In [ ]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Webcam"],
    "price": [1200000, 25000, 45000, 350000, 75000],
    "cost": [900000, 15000, 30000, 250000, 50000],
    "quantity": [3, 20, 12, 5, 8],
})

sales["revenue"] = sales["price"] * sales["quantity"]
sales["profit"] = (sales["price"] - sales["cost"]) * sales["quantity"]
sales["margin"] = (sales["profit"] / sales["revenue"] * 100).round(1)
sales["profit_grade"] = np.where(sales["margin"] >= 35, "High Margin", "Standard")
sales["revenue_tier"] = pd.cut(
    sales["revenue"],
    bins=[0, 500000, 2000000, float("inf")],
    labels=["Small", "Medium", "Large"]
)

print(sales[["product", "revenue", "profit", "margin", "profit_grade", "revenue_tier"]])

# Reading it: Laptop has the biggest revenue (Large tier) but only a 25% margin (Standard).
# Mouse has small revenue (Small tier) but a 40% margin (High Margin) -- revenue size and
# margin quality are two completely different signals, and this table shows both at once.
# 읽는 법: Laptop은 매출이 가장 크지만(Large) 마진율은 25%로 낮음(Standard). Mouse는
# 매출 규모가 작지만(Small) 마진율은 40%로 높음(High Margin) -- 매출 크기와 마진 품질은
# 완전히 다른 두 신호이며, 이 표는 둘을 한 번에 보여줌.

---
# ⚠️ Common Mistakes

### Mistake 1 — Reaching for `apply()` out of habit
**English:** `df["col"].apply(lambda x: x * 2)` works, but it processes one value at a time in a Python-level loop — roughly 100x slower than the vectorized `df["col"] * 2` on a large table, as shown directly in Example 4.  
**한글:** `df["col"].apply(lambda x: x * 2)`는 동작은 하지만, 파이썬 레벨 반복문으로 한 번에 하나씩 처리합니다 — 큰 테이블에서는 Example 4에서 직접 보였듯 벡터화된 `df["col"] * 2`보다 대략 100배 느립니다.

**✅ Fix / 해결법:**  
Reach for `apply()` only when the logic genuinely can't be expressed as vectorized math or a built-in method (`np.where`, `.str.*`, `pd.cut`) — check those first.  
`apply()`는 로직을 벡터 연산이나 내장 메서드(`np.where`, `.str.*`, `pd.cut`)로 정말 표현할 수 없을 때만 사용하세요 — 그것들부터 먼저 확인하세요.

### Mistake 2 — Ordering `np.select()` conditions incorrectly
**English:** `np.select()` checks conditions **top to bottom** and stops at the first match. Listing `score >= 60` before `score >= 90` means a 95-point score matches the *first* (least specific) condition and gets labeled `"Pass"` instead of `"A"` — the more specific `>= 90` condition never even gets checked.  
**한글:** `np.select()`는 조건을 **위에서 아래로** 검사하고 첫 번째로 일치하는 곳에서 멈춥니다. `score >= 90`보다 `score >= 60`을 먼저 나열하면, 95점은 *첫 번째*(가장 덜 구체적인) 조건에 걸려서 `"A"`가 아니라 `"Pass"`로 표시됩니다 — 더 구체적인 `>= 90` 조건은 아예 검사되지도 않습니다.

**✅ Fix / 해결법:**  
Always order `np.select()` conditions from **most specific to least specific** (highest threshold first, when checking "at least X").  
`np.select()` 조건은 항상 **가장 구체적인 것부터 가장 덜 구체적인 것 순서**로 배치하세요("최소 X 이상"을 확인할 때는 가장 높은 기준부터).

### Mistake 3 — Assuming `map()` behaves like `replace()`
**English:** `series.map(some_dict)` turns **any value not found in the dict into `NaN`** — silent, easy to miss, and it can quietly corrupt a `.sum()` or a chart downstream. `series.replace(some_dict)` looks similar but keeps unmapped values unchanged instead.  
**한글:** `series.map(some_dict)`는 **dict에 없는 값을 전부 `NaN`으로** 바꿉니다 — 조용하고 놓치기 쉬우며, 이후의 `.sum()`이나 차트를 소리 없이 망가뜨릴 수 있습니다. `series.replace(some_dict)`는 비슷해 보이지만 매핑되지 않은 값은 그대로 유지합니다.

**✅ Fix / 해결법:**  
After using `.map()`, check `.isna().sum()` on the result to see how many values weren't found — then decide with `.fillna()` whether that should become `"Other"`, `0`, or something else entirely.  
`.map()`을 쓴 뒤에는 결과에 `.isna().sum()`을 확인해서 몇 개의 값이 매핑되지 않았는지 보고, `.fillna()`로 그것이 `"Other"`, `0`, 또는 다른 무언가가 되어야 할지 결정하세요.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Default to `assign()` whenever a new column is one link in a larger method chain (filter → assign → sort) — reserve direct `df["col"] = ...` for quick, standalone scripts.  
새 열이 더 큰 메서드 체인(필터 → assign → 정렬)의 한 단계라면 `assign()`을 기본으로 사용하세요 — 직접 `df["col"] = ...`는 빠르고 단독적인 스크립트에만 남겨두세요.
- Reach for `np.where()` for two outcomes and `np.select()` for three or more — both are vectorized and dramatically faster than `apply()` with an if/else.  
결과가 두 가지면 `np.where()`, 세 가지 이상이면 `np.select()`를 사용하세요 — 둘 다 벡터화되어 있어 if/else가 있는 `apply()`보다 훨씬 빠릅니다.
- `apply(axis=1)` is one of the *slowest* common pandas operations — only reach for it when the logic genuinely needs several columns of the SAME row at once, and no vectorized alternative exists.  
`apply(axis=1)`은 pandas에서 흔히 쓰이는 것 중 *가장 느린* 축에 속합니다 — 같은 행의 여러 열이 정말로 동시에 필요하고, 벡터화된 대안이 없을 때만 사용하세요.
- `get_dummies(..., drop_first=True)` matters before feeding data into a statistical model (avoids multicollinearity) — it's not needed for a simple pivot-style breakdown you'll only look at, not model.  
`get_dummies(..., drop_first=True)`는 통계 모델에 데이터를 넣기 전에 중요합니다(다중공선성 방지) — 그냥 살펴보기만 할 피벗 스타일 분석에는 필요하지 않습니다.

---
# 🔗 Related Concepts

```
Data Cleaning                     (Section 5 -- you now have clean, correctly-typed columns to build ON)
    ↓
Column Creation & Transformation    ← you are here / 지금 여기 (Section 6)
    ↓
GroupBy                           (Section 7 -- the columns you just built become groupby() keys and agg() targets)
    ↓
Merge                             (Section 8)
    ↓
... Pivot -> Time Series -> BA Techniques
```

*How is today's topic connected to other concepts?*

**English:** Section 5's cleaning tools (`where()` / `mask()`) and today's `np.where()` share the exact same condition syntax — the difference is purpose: cleaning *replaces* values in an existing column, while today's tools *build a brand new one*. Looking ahead, a `spend_tier` column built with `pd.qcut()` today becomes tomorrow's `groupby("spend_tier")` key in Section 7 — nearly every GroupBy in real analysis groups by a column that was *derived* here, not one that arrived in the raw file.

**한글:** 5번 섹션의 정제 도구(`where()` / `mask()`)와 오늘의 `np.where()`는 정확히 같은 조건 문법을 공유합니다 — 차이는 목적입니다: 정제는 기존 열의 값을 *교체*하고, 오늘 배운 도구들은 *완전히 새로운 열을 만듭니다*. 앞을 내다보면, 오늘 `pd.qcut()`으로 만든 `spend_tier` 열은 내일 7번 섹션에서 `groupby("spend_tier")`의 키가 됩니다 — 실제 분석에서 거의 모든 GroupBy는 원본 파일에 있던 열이 아니라 여기서 *파생된* 열을 기준으로 그룹화합니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오**

**English:** A marketing manager wants a "customer value" view built from raw order totals: a readable region name instead of a code, a spend tier for segmentation, and a VIP flag for the loyalty campaign — all derived from the same source table.

**한글:** 마케팅 매니저가 원본 주문 총액 데이터로부터 "고객 가치" 뷰를 원합니다: 코드 대신 읽기 쉬운 지역 이름, 세분화를 위한 소비 등급, 로열티 캠페인을 위한 VIP 플래그 — 전부 같은 원본 테이블에서 파생됩니다.

**To Do / 할 일**
- [x] Translate region codes into readable names with `map()` / `map()`으로 지역 코드를 읽기 쉬운 이름으로 변환하기
- [x] Bucket customers into spend tiers with `qcut()` / `qcut()`으로 고객을 소비 등급으로 나누기
- [x] Flag VIP customers with `np.where()` / `np.where()`로 VIP 고객 표시하기
- [x] Chain everything with `assign()` and sort by spend / `assign()`으로 전부 체이닝하고 spend 기준으로 정렬하기

In [13]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004", "C005", "C006"],
    "region_code": ["SEO", "PUS", "SEO", "ICN", "PUS", "SEO"],
    "total_spend": [1250000, 480000, 95000, 2100000, 310000, 670000],
})

region_name = {"SEO": "Seoul", "PUS": "Busan", "ICN": "Incheon"}

result = (
    customers
    .assign(
        region = lambda x: x["region_code"].map(region_name),
        spend_tier = lambda x: pd.qcut(x["total_spend"], q=3, labels=["Bronze", "Silver", "Gold"]),
        is_vip = lambda x: np.where(x["total_spend"] >= 1000000, True, False),
    )
    .sort_values("total_spend", ascending=False)
)
print(result)

  customer_id region_code  total_spend   region spend_tier  is_vip
3        C004         ICN      2100000  Incheon       Gold    True
0        C001         SEO      1250000    Seoul       Gold    True
5        C006         SEO       670000    Seoul     Silver   False
1        C002         PUS       480000    Busan     Silver   False
4        C005         PUS       310000    Busan     Bronze   False
2        C003         SEO        95000    Seoul     Bronze   False


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**English**
Column creation and transformation is how a table gains exactly the fields an analysis needs. Direct assignment (`df["col"] = ...`) and `assign()` both add computed columns, differing mainly in whether they mutate in place or chain cleanly. `np.where()` branches a value in two, `np.select()` branches it in three or more (checked top to bottom, first match wins), and `pd.cut()` / `pd.qcut()` bucket a continuous number into fixed or quantile-based ranges. `apply()` handles logic no vectorized method can express, at a real speed cost, while `.map()` / `.replace()` translate codes into labels (differing in how they treat unmapped values). `pd.get_dummies()` one-hot encodes categories, and `pipe()` / `eval()` offer two different ways to keep a transformation chain readable. Nearly every column built here becomes tomorrow's GroupBy key or chart axis.

**한글**
열 생성과 값 변환은 테이블이 분석에 필요한 바로 그 필드를 갖추게 하는 방법입니다. 직접 대입(`df["col"] = ...`)과 `assign()`은 둘 다 계산된 열을 추가하는데, 주로 즉시 수정하는지 아니면 깔끔하게 체이닝되는지에서 차이가 납니다. `np.where()`는 값을 둘로 분기하고, `np.select()`는 셋 이상으로 분기하며(위에서 아래로 검사, 첫 일치가 승리), `pd.cut()` / `pd.qcut()`은 연속된 숫자를 고정 또는 분위수 기반 구간으로 나눕니다. `apply()`는 벡터화된 메서드로 표현할 수 없는 로직을 실제 속도 비용을 감수하고 처리하며, `.map()` / `.replace()`는 코드를 라벨로 변환합니다(매핑되지 않은 값을 다루는 방식이 다름). `pd.get_dummies()`는 카테고리를 원-핫 인코딩하고, `pipe()` / `eval()`은 변환 체인을 읽기 쉽게 유지하는 두 가지 다른 방법을 제공합니다. 여기서 만든 열은 거의 전부 내일의 GroupBy 키나 차트 축이 됩니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Column creation turns "the raw fields I was given" into "the derived fields the analysis actually needs" — a computed metric, a branched category, a binned range, or a translated label — almost always in one vectorized line rather than a manual loop.

> 열 생성은 "받은 원시 필드"를 "분석에 실제로 필요한 파생 필드"로 바꿉니다 — 계산된 지표, 분기된 카테고리, 구간화된 범위, 또는 변환된 라벨 — 거의 항상 직접 반복문이 아니라 한 줄의 벡터 연산으로.

---
# ❓ Review Questions

**Q1.** What's the key difference between `df["col"] = ...` and `df.assign(col=...)`, and when does that difference actually matter?
**Q1.** `df["col"] = ...`와 `df.assign(col=...)`의 핵심 차이는 무엇이며, 그 차이는 언제 실제로 중요해지나요?

**Q2.** When would you reach for `np.select()` instead of `np.where()`?
**Q2.** `np.where()` 대신 `np.select()`를 언제 사용해야 하나요?

**Q3.** What's the difference between how `pd.cut()` and `pd.qcut()` decide where the bin boundaries go?
**Q3.** `pd.cut()`과 `pd.qcut()`이 구간 경계를 정하는 방식은 어떻게 다른가요?

**Q4.** Why is `apply(axis=1)` usually a last resort rather than a first choice?
**Q4.** `apply(axis=1)`이 왜 보통 첫 번째 선택지가 아니라 최후의 수단인가요?

**Q5.** `series.map(some_dict)` and `series.replace(some_dict)` can look interchangeable — what happens differently when a value isn't in the dict?
**Q5.** `series.map(some_dict)`와 `series.replace(some_dict)`는 바꿔 써도 될 것처럼 보이는데, 값이 dict에 없을 때 실제로 어떻게 다르게 동작하나요?

---
*📅 Try answering these again in a few days. / 며칠 뒤에 다시 답해보세요.*